# M29 — Open-Set Baseline Suite on the M12 Backbone**Model ID:** M29 (new — see §"Why a new ID" below) &nbsp;|&nbsp; **Member:** A &nbsp;|&nbsp; **Author:** Asif**Requires:** M2 / M12 (the frozen backbone). **Nothing from Members B, C or D.**---### What this isFour standard open-set-recognition scores computed on the **M12-selected backbone's** embeddings,evaluated on ICBHI's real known-vs-unknown patient split:| Score | Definition | Why it's here ||---|---|---|| **MSP** | `1 − max softmax` | The canonical OOD baseline (Hendrycks & Gimpel) || **Entropy** | `H(softmax)` | Uncertainty of the sound-event head || **Energy** | `−logsumexp(logits)` | Energy-based OOD; usually beats MSP || **Mahalanobis (class)** | min distance to a class-conditional Gaussian | Standard feature-space OOD || **Mahalanobis (patient)** | distance to the known-patient embedding distribution | Patient-level variant |### Why this is the right next experiment`Novelty Search v2.md` Phase 2, **Attack 1** states the harshest plausible review of this project:> *"The 'novelty' here reduces to thresholding the distance between two softmax vectors… The paper> does not show that this signal is better than simpler baselines (max-softmax score, Mahalanobis> distance, or even entropy of a single head). Without that comparison, the cross-task disagreement> claim is unfounded."*>> **Mitigation needed:** ablation against energy-based OOD score, Mahalanobis distance from training> embeddings, and post-hoc entropy — not just OpenMax.That mitigation is exactly this notebook. It is also the **only unknown-detection experiment in theproject that can currently be run on real data**: the audit(`Asif's/audit/PROJECT_AUDIT.md`) found that M13/M15/M17/M19 all operate on synthetically generatedtensors, so no real cross-task result exists yet to compare against. These baselines establish thefloor that M15 must clear when it is eventually built properly.### What the outcome means either way- **A trivial baseline scores well (AUROC ≳ 0.7)** → cross-task disagreement must beat *this*, not  just OpenMax. The bar moves, and the team learns it now rather than at review.- **Everything sits near chance** → the bottleneck is the evaluation setup (n=19 unknown patients,  patient-level aggregation, class imbalance), not the mechanism. That reframes the paper toward the  contribution `Novelty Search v2.md` already calls the strongest: **the statistically defensible  evaluation redesign**. A well-characterised negative is publishable; an untested claim is not.Either way this produces the project's **first real unknown-detection numbers**, and a defensiblecomparison point against M6 — which is the one real downstream result the audit trusts(`open_set.auroc = 0.4516`, `unknown_recall = 0.0255`).### Why a new ID`Model_Training_Reference.md` has no slot for a trivial-baseline suite; M6 covers OpenMax only.M29 is proposed as an addition to the `rejection_method` ablation group (M6 vs M15 vs **M29**).Add it to the reference's Quick Index when you commit these results.---### Protocol compliance- §1 patient-independent split — **asserted**, and the unknown group is never used for fitting- §2 preprocessing — inherited unchanged from M2, so the embeddings are the M12 backbone's- §3 metric suite + open-set AUROC/AUPR/precision/recall- §4 + §4.1 `results_M29.json`, ablation group `rejection_method`- §11 `weights_only=False`, scalar casting, `eval_only` support

---## Section 1 — Environment Setup

In [ ]:
# ============================================================
# CELL 0 — ENVIRONMENT & PATHS
# ============================================================
import os, sys, platform

print("=" * 72)
print("M29 OPEN-SET BASELINE SUITE — ENVIRONMENT")
print("=" * 72)
print(f"Python : {sys.version.split()[0]}  ({platform.platform()})")

try:
    import torch
    print(f"PyTorch: {torch.__version__}  | CUDA: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU    : {torch.cuda.get_device_name(0)}")
except ImportError:
    print("ERROR: PyTorch not installed.")

IN_COLAB = "google.colab" in sys.modules
DRIVE_MOUNTED = False
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_MOUNTED = True
        print("Drive mounted.")
    except Exception as e:
        print(f"Drive mount skipped ({e})")

if IN_COLAB and DRIVE_MOUNTED:
    BASE_DIR = "/content/drive/MyDrive/OWMTL/M29"
elif IN_COLAB:
    BASE_DIR = "/content/OWMTL/M29"
elif os.path.exists("/kaggle/working"):
    BASE_DIR = "/kaggle/working"
else:
    BASE_DIR = "./outputs_M29"

RESULTS_DIR = os.path.join(BASE_DIR, "results")
CACHE_DIR = ("/content/owmtl_spec_cache" if IN_COLAB else
             "/kaggle/working/owmtl_spec_cache" if os.path.exists("/kaggle/working")
             else "./owmtl_spec_cache")
for d in (RESULTS_DIR, CACHE_DIR):
    os.makedirs(d, exist_ok=True)

print(f"\nResults : {RESULTS_DIR}")
print(f"Cache   : {CACHE_DIR}   (shared with M2/M3 — identical §2 preprocessing)")
print("=" * 72)

---## Section 2 — ConfigurationPreprocessing is inherited verbatim from M2 so the embeddings are genuinely the M12 backbone's. Anychange here silently invalidates the comparison, which is why the values are restated explicitlyrather than imported.

In [ ]:
# ============================================================
# CELL 1 — DEPENDENCIES & CONFIGURATION
# ============================================================
import subprocess, math

def pip_install(pkg, import_name=None):
    try:
        __import__(import_name or pkg.replace("-", "_"))
    except ImportError:
        print(f"Installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for p, n in [("librosa", None), ("soundfile", None), ("scikit-learn", "sklearn"),
             ("tqdm", None), ("matplotlib", None), ("seaborn", None)]:
    pip_install(p, n)
print("Dependencies ready.\n")

# ------------------------------------------------------------
# Dataset + checkpoint discovery
# ------------------------------------------------------------
import glob

AUDIO_CANDIDATES = [
    "/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files",
    "/content/drive/MyDrive/respiratory-sound-database/audio_and_txt_files",
    "/kaggle/input/respiratory-sound-database/Respiratory_Sound_Database/"
    "Respiratory_Sound_Database/audio_and_txt_files",
    "./data/audio_and_txt_files",
]
DATA_ROOT = next((p for p in AUDIO_CANDIDATES if os.path.exists(p)), None)

def find_first(patterns):
    for pat in patterns:
        hits = sorted(glob.glob(pat, recursive=True))
        if hits:
            return hits[0]
    return None

# ICBHI ships the patient->diagnosis map as patient_diagnosis.csv (vbookshelf mirror)
DIAG_FILE = find_first([
    "/content/**/patient_diagnosis.csv",
    "/content/**/*diagnosis*.csv",
    "/content/**/*diagnosis*.txt",
    "/kaggle/input/**/patient_diagnosis.csv",
    "./**/patient_diagnosis.csv",
])

# The M12-selected backbone checkpoint (M2's best_model.pth)
M2_CKPT = find_first([
    "/content/drive/MyDrive/OWMTL/M2/checkpoints/best_model.pth",
    "/content/drive/MyDrive/OWMTL/M2/best_model.pth",
    "/content/best_model.pth",
    "/kaggle/input/**/best_model.pth",
    "./**/M2/best_model.pth",
])

print(f"DATA_ROOT : {DATA_ROOT}")
print(f"DIAG_FILE : {DIAG_FILE}")
print(f"M2_CKPT   : {M2_CKPT}")
if not all([DATA_ROOT, DIAG_FILE, M2_CKPT]):
    print("\nMissing input(s). In Colab:")
    print("  ICBHI    : !kaggle datasets download -d vbookshelf/respiratory-sound-database "
          "-p /content --unzip")
    print("  M2 ckpt  : upload Asif's/M2/best_model.pth to /content/ or Drive")

CFG = {
    # ── §2 preprocessing — MUST match M2 exactly or the embeddings are not M12's ──
    "sample_rate": 16000, "duration_s": 8.0, "n_mels": 128, "n_fft": 1024,
    "hop_length": 160, "win_length": 400, "f_min": 50, "f_max": 2000,
    "n_samples": int(16000 * 8.0), "n_frames": None,

    # ── ICBHI open-set definition (Model_Training_Reference.md:182, :190) ──
    "known_classes": ["COPD", "Healthy", "URTI"],            # 64 + 26 + 14 = 104 patients
    "unknown_classes": ["Bronchiectasis", "Pneumonia", "Bronchiolitis"],   # 7 + 6 + 6 = 19
    # Asthma (1) and LRTI (2) are excluded: too few to place in either group without
    # distorting it, and the reference's open-set protocol names neither. Documented
    # in the results JSON rather than silently dropped.
    "excluded_classes": ["Asthma", "LRTI"],

    "sound_event_classes": ["Normal", "Crackle", "Wheeze", "Both"],
    "num_classes": 4,

    # ── Evaluation ──
    "known_test_fraction": 0.40,   # patient-independent split of the KNOWN group
    "batch_size": 32, "num_workers": 2, "seed": 42,
    "target_known_tpr": 0.95,      # operating point for precision/recall reporting

    "data_root": DATA_ROOT, "diag_file": DIAG_FILE, "m2_ckpt": M2_CKPT,
    "results_dir": RESULTS_DIR, "cache_dir": CACHE_DIR,
    "model_id": "M29", "member": "A", "member_name": "Asif",
}
CFG["n_frames"] = 1 + math.floor(CFG["n_samples"] / CFG["hop_length"])   # 801

print("\n" + "=" * 60)
for k, v in CFG.items():
    print(f"  {k:<22}: {v}")
print("=" * 60)

In [ ]:
# ============================================================
# CELL 2 — IMPORTS & SEED
# ============================================================
import json, time, random, warnings, tempfile, datetime
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, accuracy_score, f1_score,
                             confusion_matrix)


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(CFG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
sns.set_style("whitegrid")


class NumpyEncoder(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, np.integer): return int(o)
        if isinstance(o, np.floating): return float(o)
        if isinstance(o, np.bool_): return bool(o)
        if isinstance(o, np.ndarray): return o.tolist()
        return super().default(o)


print(f"Device: {DEVICE} ({GPU_NAME})  | seed {CFG['seed']}")

---## Section 3 — Data: cycles, patients, and the known/unknown splitTwo things must be true for this experiment to mean anything, and both are asserted rather thanassumed:1. **No patient appears in both the fitting set and the evaluation set** (protocol §1).2. **The unknown group is never used for fitting** — not for the Gaussians, not for the threshold.   `Model_Training_Reference.md:190` is explicit: the pooled unknown group is *"for evaluation only,   never for fitting."*

In [ ]:
# ============================================================
# CELL 3 — PARSE ICBHI: CYCLES + PATIENT DIAGNOSES
# ============================================================

def parse_annotation_file(txt_path):
    """One ICBHI annotation file -> list of cycles with 4-class sound-event labels."""
    cycles = []
    with open(txt_path) as f:
        for line in f:
            parts = line.split()
            if len(parts) < 4:
                continue
            try:
                start, end = float(parts[0]), float(parts[1])
                crackle, wheeze = int(parts[2]), int(parts[3])
            except ValueError:
                continue
            if end <= start:
                continue
            label = (0 if (crackle == 0 and wheeze == 0) else
                     1 if (crackle == 1 and wheeze == 0) else
                     2 if (crackle == 0 and wheeze == 1) else 3)
            cycles.append({"start": start, "end": end, "label": label})
    return cycles


def load_diagnoses(diag_path):
    """patient_id -> diagnosis string. Handles the .csv and whitespace .txt variants."""
    diag = {}
    with open(diag_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(",") if "," in line else line.split()
            if len(parts) < 2:
                continue
            try:
                diag[int(parts[0])] = parts[1].strip()
            except ValueError:
                continue          # header row
    return diag


def build_dataframe(data_root, diag_path, cfg):
    diag = load_diagnoses(diag_path)
    print(f"Loaded {len(diag)} patient diagnoses")
    counts = pd.Series(list(diag.values())).value_counts()
    print("\nDiagnosis distribution (patients):")
    for name, n in counts.items():
        grp = ("KNOWN" if name in cfg["known_classes"] else
               "UNKNOWN" if name in cfg["unknown_classes"] else "excluded")
        print(f"  {name:<18} {n:>4}   [{grp}]")

    rows = []
    for wav in sorted(glob.glob(os.path.join(data_root, "*.wav"))):
        stem = os.path.splitext(os.path.basename(wav))[0]
        txt = os.path.join(data_root, stem + ".txt")
        if not os.path.exists(txt):
            continue
        try:
            pid = int(stem.split("_")[0])
        except ValueError:
            continue
        dx = diag.get(pid)
        if dx is None:
            continue
        if dx in cfg["known_classes"]:
            grp = "known"
        elif dx in cfg["unknown_classes"]:
            grp = "unknown"
        else:
            continue                      # Asthma / LRTI — documented exclusion
        for c in parse_annotation_file(txt):
            rows.append({"wav_path": wav, "patient_id": pid, "diagnosis": dx,
                         "group": grp, **c})

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError("No cycles parsed — check DATA_ROOT and DIAG_FILE.")
    return df


print(f"Parsing ICBHI from {CFG['data_root']}\n")
df = build_dataframe(CFG["data_root"], CFG["diag_file"], CFG)

pat = df.groupby("patient_id").agg(group=("group", "first"),
                                   diagnosis=("diagnosis", "first"),
                                   n_cycles=("label", "size")).reset_index()
known_pat = pat[pat.group == "known"]
unknown_pat = pat[pat.group == "unknown"]

print(f"\nCycles      : {len(df)}")
print(f"Known       : {len(known_pat):>3} patients / {int(known_pat.n_cycles.sum()):>5} cycles")
print(f"Unknown     : {len(unknown_pat):>3} patients / {int(unknown_pat.n_cycles.sum()):>5} cycles")

# ---- patient-independent split of the KNOWN group (unknowns are all evaluation) ----
rng = np.random.RandomState(CFG["seed"])
kp = known_pat.patient_id.values.copy()
rng.shuffle(kp)
n_test = max(1, int(round(len(kp) * CFG["known_test_fraction"])))
known_test_ids = set(kp[:n_test].tolist())
known_fit_ids = set(kp[n_test:].tolist())

# ---- PROTOCOL §1 + open-set hygiene, asserted ----
assert not (known_fit_ids & known_test_ids), "Patient leakage between fit and test."
assert not (set(unknown_pat.patient_id) & known_fit_ids), \
    "An unknown patient is in the fitting set — the unknown group must never be fitted on."
assert not (set(unknown_pat.patient_id) & known_test_ids), "Group overlap."
print(f"\n[OK] §1 verified: fit {len(known_fit_ids)} / test {len(known_test_ids)} known patients, "
      f"disjoint; {len(unknown_pat)} unknown patients used for EVALUATION ONLY.")

df["split"] = df.patient_id.map(
    lambda p: "fit" if p in known_fit_ids else ("known_test" if p in known_test_ids
                                                else "unknown_test"))
print(df.groupby("split").size().to_string())

In [ ]:
# ============================================================
# CELL 4 — LOG-MEL EXTRACTION (identical to M2) + CACHE
# ============================================================
# Preprocessing MUST be byte-identical to M2's, otherwise these are not the
# M12 backbone's embeddings and the whole experiment is mis-specified.

def extract_log_mel(wav_path, start, end, cfg):
    sr, n_samples = cfg["sample_rate"], cfg["n_samples"]
    try:
        audio, _ = librosa.load(wav_path, sr=sr, offset=start,
                                duration=max(end - start, 0.05), mono=True)
    except Exception as e:
        # A silent all-zero spectrogram here would be trained on and
        # scored as a real cycle. Fail instead of substituting
        # (Model_Training_Protocol.md section 1.2).
        raise RuntimeError(f"failed to load audio: {wav_path}") from e
    if len(audio) == 0:
        # Empty decode is a failed read, not a silent zero cycle.
        raise RuntimeError(f"empty audio decoded from audio: {wav_path}")
    if len(audio) < n_samples:
        audio = np.tile(audio, math.ceil(n_samples / len(audio)))[:n_samples]
    else:
        audio = audio[:n_samples]

    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=cfg["n_mels"], n_fft=cfg["n_fft"],
        hop_length=cfg["hop_length"], win_length=cfg["win_length"],
        fmin=cfg["f_min"], fmax=cfg["f_max"], power=2.0)
    lm = librosa.power_to_db(mel, ref=np.max)
    lm = (lm - lm.min()) / (lm.max() - lm.min() + 1e-8)
    T = lm.shape[1]
    lm = (np.pad(lm, ((0, 0), (0, cfg["n_frames"] - T)), mode="constant")
          if T < cfg["n_frames"] else lm[:, :cfg["n_frames"]])
    return lm[np.newaxis].astype(np.float32)


class CycleDataset(Dataset):
    def __init__(self, frame, cfg):
        self.df = frame.reset_index(drop=True)
        self.cfg = cfg

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        spec = extract_log_mel(r["wav_path"], r["start"], r["end"], self.cfg)
        return (torch.from_numpy(spec), int(r["label"]), int(r["patient_id"]))


loader = DataLoader(CycleDataset(df, CFG), batch_size=CFG["batch_size"], shuffle=False,
                    num_workers=CFG["num_workers"], pin_memory=torch.cuda.is_available())
print(f"Batches: {len(loader)}  ({len(df)} cycles)")
_x, _y, _p = next(iter(loader))
print(f"Batch spec {tuple(_x.shape)} | expected (B, 1, {CFG['n_mels']}, {CFG['n_frames']})")
assert _x.shape[1:] == (1, CFG["n_mels"], CFG["n_frames"]), "Spectrogram shape mismatch vs M2."

---## Section 4 — Load the frozen M12 backboneThe architecture is read from the checkpoint's own `model_config`, not hard-coded, so this celltracks whatever M12 actually selected. It also re-verifies the checkpoint's recorded score, the sameintegrity check M12 performs.

In [ ]:
# ============================================================
# CELL 5 — M2 / M12 BACKBONE (architecture read from the checkpoint)
# ============================================================

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=(2, 2)):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True), nn.MaxPool2d(pool))

    def forward(self, x):
        return self.block(x)


class M2_CNN(nn.Module):
    """Identical definition to Asif's/M2 — required to load its state_dict."""

    def __init__(self, num_classes=4, depth=4, base_width=32, dropout=0.5, fc_dim=128):
        super().__init__()
        channels = [base_width * (2 ** i) for i in range(depth)]
        blocks, in_ch = [], 1
        for out_ch in channels:
            blocks.append(ConvBlock(in_ch, out_ch)); in_ch = out_ch
        self.encoder = nn.Sequential(*blocks)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(nn.Linear(channels[-1], fc_dim),
                                  nn.ReLU(inplace=True), nn.Linear(fc_dim, num_classes))
        self.embedding_dim = channels[-1]

    def forward(self, x):
        return self.head(self.dropout(self.gap(self.encoder(x)).flatten(1)))

    def get_embedding(self, x):
        return self.gap(self.encoder(x)).flatten(1)


state = torch.load(CFG["m2_ckpt"], map_location=DEVICE, weights_only=False)   # §11.A
mcfg = state.get("model_config", {"depth": 5, "base_width": 48, "dropout": 0.4})
print(f"Checkpoint model_config : {mcfg}")
print(f"Checkpoint epoch        : {state.get('epoch')}")
print(f"Checkpoint best_score   : {state.get('best_score')}")

backbone = M2_CNN(num_classes=CFG["num_classes"], depth=int(mcfg["depth"]),
                  base_width=int(mcfg["base_width"]),
                  dropout=float(mcfg.get("dropout", 0.4))).to(DEVICE)
missing, unexpected = backbone.load_state_dict(state["model_state"], strict=False)
assert not missing, f"Checkpoint is missing weights for: {missing[:5]}"
backbone.eval()

n_params = sum(p.numel() for p in backbone.parameters())
print(f"\nBackbone loaded: {n_params:,} params, embedding dim {backbone.embedding_dim}")
if unexpected:
    print(f"  (ignored {len(unexpected)} unexpected keys)")
print("This is the frozen M12 backbone — no fine-tuning happens in this notebook.")

In [ ]:
# ============================================================
# CELL 6 — EXTRACT LOGITS + EMBEDDINGS FOR EVERY CYCLE
# ============================================================
all_logits, all_embs, all_pids, all_labels = [], [], [], []

t0 = time.time()
with torch.no_grad():
    for specs, labels, pids in tqdm(loader, desc="Extracting"):
        specs = specs.to(DEVICE, non_blocking=True)
        emb = backbone.get_embedding(specs)
        logits = backbone.head(emb)          # dropout is inactive in eval()
        all_logits.append(logits.cpu().numpy())
        all_embs.append(emb.cpu().numpy())
        all_pids.append(pids.numpy())
        all_labels.append(labels.numpy())

LOGITS = np.concatenate(all_logits)
EMBS = np.concatenate(all_embs)
PIDS = np.concatenate(all_pids)
LABELS = np.concatenate(all_labels)
print(f"\nExtracted {LOGITS.shape[0]} cycles in {time.time() - t0:.0f}s | "
      f"logits {LOGITS.shape}, embeddings {EMBS.shape}")

SPLIT = df["split"].values
assert len(SPLIT) == len(LOGITS), "Row alignment broken between df and extracted tensors."

---## Section 5 — The four baseline scoresEvery score is **fit on known-fit patients only** and produces a value where *higher means morelikely unknown*. Cycle-level scores are aggregated to the patient by mean, because the task ispatient-level diagnosis — this is also the level `Model_Training_Reference.md:266` specifies, andthe level at which M15 should have been evaluated but was not.

In [ ]:
# ============================================================
# CELL 7 — SCORE FUNCTIONS
# ============================================================

def softmax_np(x):
    x = x - x.max(axis=1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)


def score_msp(logits, **_):
    """1 - max softmax probability (Hendrycks & Gimpel). Higher = more unknown."""
    return 1.0 - softmax_np(logits).max(axis=1)


def score_entropy(logits, **_):
    """Shannon entropy of the softmax, normalised to [0, 1]."""
    p = softmax_np(logits)
    h = -(p * np.log(p + 1e-12)).sum(axis=1)
    return h / np.log(p.shape[1])


def score_energy(logits, **_):
    """-logsumexp(logits) -- the energy-based OOD score (Liu et al.)."""
    m = logits.max(axis=1, keepdims=True)
    lse = (m.squeeze(1) + np.log(np.exp(logits - m).sum(axis=1)))
    return -lse


def fit_mahalanobis_class(embs_fit, labels_fit, n_classes):
    """
    Class-conditional Gaussians with a shared covariance, the standard
    feature-space OOD detector. Fitted on known-fit cycles only.
    """
    means, centered = [], []
    for c in range(n_classes):
        m = embs_fit[labels_fit == c]
        if len(m) < 2:
            means.append(embs_fit.mean(axis=0))
            continue
        mu = m.mean(axis=0)
        means.append(mu)
        centered.append(m - mu)
    cov = np.cov(np.vstack(centered).T) if centered else np.cov(embs_fit.T)
    cov += np.eye(cov.shape[0]) * 1e-6          # numerical stability
    return np.array(means), np.linalg.pinv(cov)


def score_mahalanobis_class(embs, means, prec):
    """Minimum Mahalanobis distance to any known class centroid."""
    d = []
    for mu in means:
        diff = embs - mu
        d.append(np.einsum("ij,jk,ik->i", diff, prec, diff))
    return np.min(np.stack(d, axis=1), axis=1)


def fit_mahalanobis_single(x_fit):
    """One Gaussian over the fitting distribution (used at patient level)."""
    mu = x_fit.mean(axis=0)
    cov = np.cov(x_fit.T) + np.eye(x_fit.shape[1]) * 1e-6
    return mu, np.linalg.pinv(cov)


def score_mahalanobis_single(x, mu, prec):
    diff = x - mu
    return np.einsum("ij,jk,ik->i", diff, prec, diff)


def aggregate_to_patient(values, pids):
    """Cycle-level -> patient-level by mean. Returns (patient_ids, scores)."""
    order = np.unique(pids)
    return order, np.array([values[pids == p].mean() for p in order])


print("Score functions defined: MSP, entropy, energy, Mahalanobis (class + patient).")

In [ ]:
# ============================================================
# CELL 8 — FIT ON KNOWN-FIT ONLY, THEN SCORE EVERY PATIENT
# ============================================================
fit_mask = SPLIT == "fit"
eval_mask = SPLIT != "fit"

print(f"Fitting on {fit_mask.sum()} known-fit cycles "
      f"({len(np.unique(PIDS[fit_mask]))} patients)")
print(f"Evaluating on {eval_mask.sum()} cycles "
      f"({len(np.unique(PIDS[eval_mask]))} patients)\n")

# --- cycle-level scores ---
cycle_scores = {
    "MSP": score_msp(LOGITS),
    "Entropy": score_entropy(LOGITS),
    "Energy": score_energy(LOGITS),
}
means_c, prec_c = fit_mahalanobis_class(EMBS[fit_mask], LABELS[fit_mask], CFG["num_classes"])
cycle_scores["Mahalanobis_class"] = score_mahalanobis_class(EMBS, means_c, prec_c)

# --- patient-level aggregation ---
eval_pids = np.unique(PIDS[eval_mask])
patient_scores = {}
for name, vals in cycle_scores.items():
    ids, agg = aggregate_to_patient(vals[eval_mask], PIDS[eval_mask])
    patient_scores[name] = dict(zip(ids.tolist(), agg.tolist()))

# --- patient-level Mahalanobis on mean embeddings (fit on known-fit patients) ---
def patient_mean_embeddings(mask):
    ids = np.unique(PIDS[mask])
    return ids, np.array([EMBS[mask][PIDS[mask] == p].mean(axis=0) for p in ids])

fit_ids, fit_pemb = patient_mean_embeddings(fit_mask)
ev_ids, ev_pemb = patient_mean_embeddings(eval_mask)
mu_p, prec_p = fit_mahalanobis_single(fit_pemb)
patient_scores["Mahalanobis_patient"] = dict(
    zip(ev_ids.tolist(), score_mahalanobis_single(ev_pemb, mu_p, prec_p).tolist()))

# --- ground truth: 1 = unknown ---
grp = dict(zip(pat.patient_id, pat.group))
y_true = np.array([1 if grp[p] == "unknown" else 0 for p in eval_pids])
print(f"Evaluation set: {int((y_true == 0).sum())} known + {int(y_true.sum())} unknown patients")
assert y_true.sum() > 0 and (y_true == 0).sum() > 0, "Evaluation set must contain both groups."

---## Section 6 — EvaluationAUROC and AUPR per score, plus precision/recall at a threshold chosen to retain 95% of knownpatients — a clinically meaningful operating point (you cannot flag a fifth of healthy patients as"unknown disease" and stay usable).The comparison row is **M6**, the only real downstream result in the project.

In [ ]:
# ============================================================
# CELL 9 — OPEN-SET METRICS
# ============================================================
M6_REFERENCE = {"auroc": 0.4516, "aupr": 0.2352,
                "unknown_precision": 0.2745, "unknown_recall": 0.0255,
                "source": "Barshon's/M6/result/results_M6.json (real data, audit-verified)"}

def evaluate_score(name, score_map):
    s = np.array([score_map[p] for p in eval_pids], dtype=float)
    auroc = roc_auc_score(y_true, s)
    aupr = average_precision_score(y_true, s)

    # threshold retaining `target_known_tpr` of KNOWN patients below it
    known_s = s[y_true == 0]
    thr = np.quantile(known_s, CFG["target_known_tpr"])
    flagged = s >= thr
    tp = int(((flagged == 1) & (y_true == 1)).sum())
    fp = int(((flagged == 1) & (y_true == 0)).sum())
    fn = int(((flagged == 0) & (y_true == 1)).sum())
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return {"score": name, "auroc": round(float(auroc), 4), "aupr": round(float(aupr), 4),
            "unknown_precision": round(prec, 4), "unknown_recall": round(rec, 4),
            "unknown_f1": round(f1, 4), "threshold": round(float(thr), 6),
            "n_known": int((y_true == 0).sum()), "n_unknown": int(y_true.sum())}


RESULTS = [evaluate_score(n, m) for n, m in patient_scores.items()]
RESULTS.sort(key=lambda r: r["auroc"], reverse=True)

print("=" * 96)
print("M29 — OPEN-SET BASELINES ON THE M12 BACKBONE (patient-level, ICBHI known vs pooled unknown)")
print("=" * 96)
print(f"{'Score':<24}{'AUROC':>9}{'AUPR':>9}{'U-Prec':>9}{'U-Rec':>9}{'U-F1':>9}")
print("-" * 96)
for r in RESULTS:
    print(f"{r['score']:<24}{r['auroc']:>9.4f}{r['aupr']:>9.4f}"
          f"{r['unknown_precision']:>9.4f}{r['unknown_recall']:>9.4f}{r['unknown_f1']:>9.4f}")
print("-" * 96)
print(f"{'M6 OpenMax (real)':<24}{M6_REFERENCE['auroc']:>9.4f}{M6_REFERENCE['aupr']:>9.4f}"
      f"{M6_REFERENCE['unknown_precision']:>9.4f}{M6_REFERENCE['unknown_recall']:>9.4f}"
      f"{'—':>9}")
print("=" * 96)
print(f"Operating point: threshold retains {CFG['target_known_tpr']:.0%} of known patients.")
print(f"Evaluation set : {RESULTS[0]['n_known']} known + {RESULTS[0]['n_unknown']} unknown patients")

# ---- interpretation, stated in the notebook so it cannot be spun later ----
best = RESULTS[0]
print("\n" + "=" * 96)
print("READING THE RESULT")
print("=" * 96)
if best["auroc"] >= 0.70:
    print(f"* `{best['score']}` reaches AUROC {best['auroc']:.4f} using NOTHING but the "
          f"sound-event backbone.\n"
          f"  Cross-task disagreement (M15) must beat THIS to claim novelty -- not just "
          f"M6's {M6_REFERENCE['auroc']:.4f}.\n"
          f"  Novelty Search v2 Attack 1 is now a live risk and must be addressed in the paper.")
elif best["auroc"] >= 0.60:
    print(f"* Best baseline `{best['score']}` = {best['auroc']:.4f}: weak but above chance.\n"
          f"  This is the real floor for M15. Any cross-task result at or below it is not a "
          f"contribution.")
else:
    print(f"* No baseline clears 0.60 (best `{best['score']}` = {best['auroc']:.4f}).\n"
          f"  With n={best['n_unknown']} unknown patients the task may be underpowered rather "
          f"than the methods inadequate.\n"
          f"  That points the paper at the evaluation-redesign contribution Novelty Search v2 "
          f"already calls the strongest, and makes a characterised negative result the honest "
          f"framing.")
if all(r["auroc"] <= 0.5 for r in RESULTS):
    print("\n* WARNING: every baseline is at or below chance. Before concluding anything, verify "
          "the backbone checkpoint loaded correctly and that group labels are not inverted.")
print("=" * 96)

In [ ]:
# ============================================================
# CELL 10 — PLOTS
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(20, 5.6))
palette = sns.color_palette("tab10", len(patient_scores))

# (a) ROC
for (name, m), c in zip(patient_scores.items(), palette):
    s = np.array([m[p] for p in eval_pids])
    fpr, tpr, _ = roc_curve(y_true, s)
    axes[0].plot(fpr, tpr, color=c, lw=2,
                 label=f"{name} (AUROC {roc_auc_score(y_true, s):.3f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=1, alpha=0.6, label="chance")
axes[0].axhline(0, color="none")
axes[0].set_xlabel("False positive rate"); axes[0].set_ylabel("True positive rate")
axes[0].set_title("ROC — unknown-patient detection"); axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# (b) Precision–recall
base_rate = y_true.mean()
for (name, m), c in zip(patient_scores.items(), palette):
    s = np.array([m[p] for p in eval_pids])
    pr, rc, _ = precision_recall_curve(y_true, s)
    axes[1].plot(rc, pr, color=c, lw=2,
                 label=f"{name} (AUPR {average_precision_score(y_true, s):.3f})")
axes[1].axhline(base_rate, ls="--", color="k", lw=1, alpha=0.6,
                label=f"base rate ({base_rate:.3f})")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision–Recall"); axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

# (c) AUROC bar vs M6
names = [r["score"] for r in RESULTS] + ["M6 OpenMax"]
vals = [r["auroc"] for r in RESULTS] + [M6_REFERENCE["auroc"]]
cols = list(palette[:len(RESULTS)]) + ["#888888"]
axes[2].bar(range(len(vals)), vals, color=cols, alpha=0.9)
axes[2].axhline(0.5, ls="--", color="red", lw=1.5, label="chance (0.5)")
axes[2].set_xticks(range(len(names)))
axes[2].set_xticklabels(names, rotation=35, ha="right", fontsize=8)
axes[2].set_ylabel("AUROC"); axes[2].set_ylim(0, 1)
axes[2].set_title("Baselines vs. M6 OpenMax"); axes[2].legend(fontsize=8)
axes[2].grid(True, axis="y", alpha=0.3)

plt.suptitle("M29 — Open-Set Baselines on the M12 Backbone (patient-level)", fontsize=13)
plt.tight_layout()
p1 = os.path.join(CFG["results_dir"], "openset_baselines.png")
plt.savefig(p1, dpi=150, bbox_inches="tight"); plt.show()
print(f"Saved: {p1}")

# (d) score distributions — shows *why* a score works or doesn't
n = len(patient_scores)
fig, axes = plt.subplots(1, n, figsize=(4.2 * n, 4))
for ax, (name, m) in zip(np.atleast_1d(axes), patient_scores.items()):
    s = np.array([m[p] for p in eval_pids])
    ax.hist(s[y_true == 0], bins=20, alpha=0.65, label="known", color="#1f77b4", density=True)
    ax.hist(s[y_true == 1], bins=20, alpha=0.65, label="unknown", color="#d62728", density=True)
    ax.set_title(name, fontsize=10); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.suptitle("Patient-level score distributions — overlap is the story", fontsize=12)
plt.tight_layout()
p2 = os.path.join(CFG["results_dir"], "score_distributions.png")
plt.savefig(p2, dpi=150, bbox_inches="tight"); plt.show()
print(f"Saved: {p2}")

---## Section 7 — Results JSON

In [ ]:
# ============================================================
# CELL 11 — EXPORT results_M29.json (§4 + §4.1)
# ============================================================
best = RESULTS[0]
results_payload = {
    "meta": {
        "model_id": "M29",
        "model_name": "Open-Set Baseline Suite on the M12 Backbone",
        "member": "A", "member_name": CFG["member_name"],
        "date_completed": datetime.date.today().isoformat(),
        "is_augmented": False, "augmentation_method": "none",
        "notes": (
            "Four standard OOD scores (MSP, entropy, energy, Mahalanobis) computed on the "
            "frozen M12 backbone's embeddings and evaluated at PATIENT level on ICBHI's real "
            "known-vs-pooled-unknown split. No training occurs; the backbone is frozen. All "
            "scorers are fit on known-fit patients only -- the 19-patient unknown group is "
            "used for evaluation exclusively (Model_Training_Reference.md:190). Exists to "
            "supply the trivial-baseline ablation demanded by Novelty Search v2 Attack 1, and "
            "to establish the real floor M15 must clear. NOTE: M29 is a new model ID not yet "
            "in Model_Training_Reference.md -- add it to the Quick Index under the "
            "rejection_method ablation group."),
    },
    "config": {
        "sample_rate": CFG["sample_rate"], "n_mels": CFG["n_mels"], "n_fft": CFG["n_fft"],
        "hop_length": CFG["hop_length"], "win_length": CFG["win_length"],
        "f_min": CFG["f_min"], "f_max": CFG["f_max"], "duration_s": CFG["duration_s"],
        "architecture": f"frozen M12 backbone ({mcfg}) + post-hoc OOD scoring",
        "backbone_checkpoint": CFG["m2_ckpt"],
        "aggregation": "cycle scores averaged to patient level",
        "operating_point": f"threshold retains {CFG['target_known_tpr']:.0%} of known patients",
        "seed": CFG["seed"],
    },
    "environment": {
        "platform": ("Google Colab" if IN_COLAB else
                     "Kaggle" if os.path.exists("/kaggle/working") else "Local"),
        "gpu_name": GPU_NAME, "pytorch_version": torch.__version__,
        "python_version": sys.version.split()[0],
    },
    "dataset_info": {
        "dataset": "ICBHI_2017",
        "known_classes": CFG["known_classes"], "unknown_classes": CFG["unknown_classes"],
        "excluded_classes": CFG["excluded_classes"],
        "excluded_rationale": ("Asthma (n=1) and LRTI (n=2) are too few to place in either "
                               "group without distorting it, and the reference's open-set "
                               "protocol names neither."),
        "n_known_patients": int(len(known_pat)), "n_unknown_patients": int(len(unknown_pat)),
        "n_fit_patients": int(len(known_fit_ids)),
        "n_known_test_patients": int(len(known_test_ids)),
        "total_cycles": int(len(df)),
        "split_method": "patient_independent_known_60_40; unknown group evaluation-only",
        "patient_leakage_verified": True,
        "unknown_group_never_fitted": True,
        "evaluation_level": "patient",
    },
    "efficiency": {
        "total_params": int(n_params), "trainable_params": 0,
        "model_size_mb": None, "training_time_total_s": 0,
        "training_time_per_epoch_s_avg": 0, "gpu_name": GPU_NAME,
        "inference_time_ms_per_sample": None,
        "note": "Backbone frozen and inherited from M12; M29 trains nothing.",
    },
    "best_epoch": {"epoch": None, "primary_metric": "auroc",
                   "primary_metric_value": best["auroc"]},
    "best_metrics": {
        "open_set": {
            "best_score_name": best["score"], "auroc": best["auroc"], "aupr": best["aupr"],
            "unknown_precision": best["unknown_precision"],
            "unknown_recall": best["unknown_recall"], "unknown_f1": best["unknown_f1"],
            "n_known_patients": best["n_known"], "n_unknown_patients": best["n_unknown"],
        },
        "all_scores": RESULTS,
        "reference_M6_openmax": M6_REFERENCE,
    },
    "ablation": {
        "ablation_group": "rejection_method",
        "ablation_role": "baseline",
        "baseline_model_id": None,
        "variable_changed": ("rejection signal: trivial post-hoc OOD scores on the "
                             "sound-event backbone, with no disease head and no cross-task "
                             "mechanism"),
        "variables_held_constant": [
            "backbone: M12_selected (frozen)", "preprocessing: identical to M2",
            "data_split: patient_independent", "augmentation: none", "seed: 42",
            "evaluation_level: patient",
        ],
        "component_flags": {
            "has_sound_event_head": True, "has_disease_head": False,
            "has_cross_task_consistency": False, "has_cqkd_regularization": False,
            "has_openmax_rejection": False, "owl_stage": 1, "compression_clusters": None,
        },
        "loss_weights": {"sound_event_weight": 1.0, "disease_weight": None,
                         "consistency_weight": None},
        "purpose": ("Mitigates Novelty Search v2 Phase 2 Attack 1: 'the paper does not show "
                    "that this signal is better than simpler baselines'. Any M15 claim must "
                    "clear these numbers, not only M6's."),
    },
    "training_history": [],
}

out = os.path.join(CFG["results_dir"], "results_M29.json")
with open(out, "w") as f:
    json.dump(results_payload, f, indent=2, cls=NumpyEncoder)
print(f"Saved: {out}")
print(f"\nBest baseline: {best['score']}  AUROC {best['auroc']:.4f}  "
      f"AUPR {best['aupr']:.4f}")
print(f"M6 OpenMax   : AUROC {M6_REFERENCE['auroc']:.4f}  AUPR {M6_REFERENCE['aupr']:.4f}")

In [ ]:
# ============================================================
# FINAL CELL — TEAM HANDOFF (§11.D)
# ============================================================
import shutil
try:
    from IPython.display import display, FileLink
    HAS_IPY = True
except ImportError:
    HAS_IPY = False

files = sorted(glob.glob(os.path.join(CFG["results_dir"], "results_M29.json")) +
               glob.glob(os.path.join(CFG["results_dir"], "*.png")))
print("=" * 62)
print("M29 OUTPUTS")
print("=" * 62)
for f in files:
    print(f"Ready: {os.path.basename(f):<28} ({os.path.getsize(f) / 1024**2:.2f} MB)")
    if HAS_IPY:
        display(FileLink(f))

if files:
    bundle = os.path.join(BASE_DIR, "bundle")
    os.makedirs(bundle, exist_ok=True)
    for f in files:
        shutil.copy2(f, os.path.join(bundle, os.path.basename(f)))
    z = shutil.make_archive(os.path.join(BASE_DIR, "M29_handoff_bundle"), "zip", bundle)
    print(f"\nZIP: {z}")
    if HAS_IPY:
        display(FileLink(z))
    if IN_COLAB:
        print(f'\n  from google.colab import files; files.download("{z}")')

print("=" * 62)
print("\nSend to Member B (M15): results_M29.json — these are the numbers the")
print("cross-task consistency scorer has to beat to be a contribution.")
print("=" * 62)

---## Section 8 — Summary & Next Steps### What this establishesThe project's **first real unknown-detection numbers**. Until now the only genuine downstream resultwas M6 (OpenMax, AUROC 0.4516) — every other open-set number in the repo came from synthetic data.M29 adds four standard baselines on real audio, at patient level, with the unknown group neverfitted on.### How to use it1. **Give `results_M29.json` to Member B.** M15's claim is not "beats OpenMax", it is "beats the   best trivial post-hoc score on the same backbone". This file defines that bar.2. **Add M29 to `Model_Training_Reference.md`'s Quick Index** under the `rejection_method` ablation   group (M6 vs M29 vs M15).3. **Put the comparison table in the paper regardless of the outcome.** Reviewer #2's Attack 1 is   answered by the table existing, not by which row wins.### Honest limits- **n = 19 unknown patients.** Every metric here has wide intervals; treat differences of a few  points as noise. This is the small-N problem the proposal already flags, and it is the reason the  evaluation-redesign contribution matters.- **The known/unknown split is one draw.** Repeating over several seeds, or LOPO on the unknown  group, would give error bars — worth doing before the numbers go in the manuscript.- **These are post-hoc scores on a backbone trained for sound events, not disease.** They are a  floor, not a ceiling. A weak result here does not by itself vindicate the cross-task mechanism.